# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR^2) Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, their fields, and fields' `@id`s.

In [ ]:
# List all RecordSets, their @ids, and fields
print("Available record sets and fields:")
for record_set in dataset.record_sets:
    print(f"- RecordSet @id: {record_set['@id']}")
    print(f"  Name: {record_set.get('name', '(unnamed)')}")
    fields = record_set.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print(f"  Fields (@id):")
    for field in fields:
        # field can be str @id or a dict with @id and other metadata
        if isinstance(field, dict):
            print(f"    - {field['@id']}   (name: {field.get('name', '')})")
        else:
            print(f"    - {field}")
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Get all record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for RecordSet {record_set_id}.")

# For this dataset, the main tabular data is typically in the first record set.
# Let's print the columns for the first DataFrame (if present):
if len(dataframes) > 0:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"\nColumns in RecordSet {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No record set dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section illustrates operations like removing outliers, transforming data distributions, or grouping data by attributes to prepare for further analysis.

**All columns and fields are referenced by `@id`.**

In [ ]:
# Identify numeric fields (@id) from dataframes, choose one for demonstration
# We'll try to infer a suitable numeric field by looking for typical numeric names. Adjust as needed!
import numpy as np

if len(dataframes) > 0:
    df = dataframes[main_record_set_id]
    
    # Try finding a field with age, interval, or similar in the @id (heuristic)
    numeric_candidates = [col for col in df.columns if any(k in col.lower() for k in ['age', 'interval', 'duration', 'years', 'month'])]
    found_numeric = False
    for candidate in numeric_candidates:
        # Try to convert to numeric and check if not all values are NaN
        try:
            df[candidate] = pd.to_numeric(df[candidate], errors='coerce')
            if df[candidate].notnull().sum() > 0:
                numeric_field_id = candidate
                found_numeric = True
                break
        except Exception:
            continue
    if found_numeric:
        # Choose a threshold (example: 50 for Age, 12 for months, etc.)
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].mean() > 0 else 1
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())
        # Normalize the field
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())
        
        # Try to group by a categorical field
        # Heuristic: look for any field with 'sex', 'msi', 'site', 'location', etc. in @id
        group_candidates = [col for col in df.columns if any(g in col.lower() for g in ['sex', 'msi', 'location', 'site', 'group'])]
        group_field = None
        if len(group_candidates) > 0:
            group_field = group_candidates[0]
        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame()
            print(f"\nGrouped mean {numeric_field_id} by {group_field}:")
            print(grouped_df.head())
        else:
            print("\nNo suitable group field found for grouping.")
    else:
        print("No suitable numeric field found for EDA.")
else:
    print("Data not loaded previously.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes) > 0 and 'numeric_field_id' in locals() and df[numeric_field_id].notnull().sum() > 0:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    
    # If grouped data exists, plot means by category
    if 'group_field' in locals() and group_field and group_field in df.columns:
        plt.figure(figsize=(8,4))
        group_means = df.groupby(group_field)[numeric_field_id].mean().dropna()
        group_means.plot(kind='bar')
        plt.title(f"Mean {numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()

## 6. Conclusion
This notebook demonstrated how to:

- Load a Croissant-schema dataset using the `mlcroissant` library
- Explore the metadata, record sets, and available fields (referenced by `@id`)
- Extract tabular data from a selected record set by `@id`
- Perform exploratory data analysis using only `@id` references for columns/fields
- Visualize distributions and groupwise statistics

Adjust your analysis and exploration to the specifics of your research question. Always use the field and record set `@id`s to ensure reproducibility and full alignment with FAIR practices.